# **Conserved Model B** — $g\,\nabla^2(\phi^2)$ (composite $\nabla$ vertex, d = 1)

A scalar density with a *conserving* (porous-medium-type) nonlinearity, driven by white noise:

$$\partial_t\phi = D\,\partial_x^2\phi - \mu\phi + g\,\partial_x^2(\phi^2) + \eta,\qquad
  \langle\eta\eta\rangle = 2T\,\delta\,\delta.$$

The $\nabla^2$ acts on the $\phi^2$ **composite**, so the vertex carries a momentum-space form
factor on the *response-leg* momentum (`mode='composite'`): $F_R\propto q^2\ell^2$, $F_K\propto q^4$.
The conservation law falls out — the tadpole gets $F=0$ and the bubble $F\propto q^2$, so
$\Sigma(q\to0)\to0$.  This is exactly the case a constant Hartree mass-shift cannot represent.

**Note:** because the dynamics conserve $\int\phi\,dx$, the equal-time variance is *suppressed*
(a weak end-to-end target); the rigorous check is the per-$q$ self-energy (validated to ~1% vs the
independent `loop_dyson` oracle in `tests/test_full_integrator.py`).

In [ ]:
import os, sys, time
# --- depth-robust repo root: walk up until we find the 'pipeline' package ---
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'pipeline')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(os.path.join(_root, 'notebooks'))  # cwd=notebooks/ so relative paths resolve
import numpy as np
import matplotlib.pyplot as plt
from pipeline.compute import compute_cumulants
from pipeline.theory import TheoryBuilder
from models.spatial_field_1d_sim import simulate, equal_time_correlator

def order_label(ell):
    """0->'tree', 1->'tree + 1-loop', 2->'tree + 1-loop + 2-loop'."""
    return ('tree' if ell == 0 else
            'tree + ' + ' + '.join('%d-loop' % j for j in range(1, ell + 1)))

mu, D, g, T = 1.0, 2.0, 0.25, 1.0         # mass, diffusion, conserved coupling, noise temp

def build_theory():
    return (TheoryBuilder('model-b-1d', n_populations=0)
            .physical_field('phi', spatial_dim=1)
            .parameter('mu', default=mu, domain='positive')
            .parameter('D',  default=D,  domain='positive')
            .parameter('g',  default=g,  domain='real')
            .parameter('T',  default=T,  domain='positive')
            .equation(lhs='(Dt + mu - D*Laplacian)*phi', rhs='g*Laplacian*phi^2')
            .set_action_text('phit*(Dt(phi) + mu*phi - D*Lap(phi) - g*Lap(phi^2)) - T*phit^2')
            .operator_ir()
            .boundary('infinite').initial('stationary').build())

## 0. Choose the order — `k` and `ℓ`

In [ ]:
# ============================  CHOOSE THE ORDER  ============================
MAX_ELL    = 1      # loop order ℓ:  0 = tree,  1 = +1-loop  (ℓ=2 is correct but slow)
K_EXTERNAL = 2      # correlator order k:  2 = two-point ⟨phiphi⟩
VERBOSE    = True   # True ⇒ print the staged [1/7]…[7/7] pipeline for each order
# ===========================================================================

if K_EXTERNAL != 2:
    raise NotImplementedError("v1 implements the k=2 two-point correlator.")

xs = np.linspace(0.0, 6.0, 25)                       # output separations x ≥ 0
kw = dict(k=K_EXTERNAL, external_fields=[('phi', 1), ('phi', 1)], spatial_grid=xs,
          tau_max=0.0, tau_step=1.0,                 # equal-time only (τ=0) — fast
          verbose=VERBOSE, use_cache=False, mf_dae_n_starts=4)
fund = {'mu': mu, 'D': D, 'g': g, 'T': T}
orders = list(range(0, MAX_ELL + 1))
print('will compute orders ℓ =', orders, ' at k =', K_EXTERNAL)

## 1. Theory — every order up to `MAX_ELL` through `compute_cumulants`

The operator-IR action `Lap(phi^2)` lowers to a **composite-derivative vertex**: the pipeline
extracts a per-diagram momentum form factor (`Lap→−k²`) and averages it over the loop momentum by
Gauss–Hermite.  `mode='composite'` is selected automatically from the vertex's base field-degree.

In [ ]:
curves = {}
for ell in orders:
    t0 = time.time()
    out = compute_cumulants(build_theory(), max_ell=ell, fundamental=fund, **kw)
    mid = out['C_tau_x'].shape[0] // 2               # τ = 0 row
    curves[ell] = np.real(out['C_tau_x'])[mid]
    si = out.get('spatial_info', {}) or {}
    print('%-26s C(0,0) = %.4f   mode = %s   live diagrams = %s   (%.0fs)'
          % (order_label(ell), curves[ell][0], si.get('vertex_mode', '—'),
             si.get('n_live_diagrams', '—'), time.time() - t0))

print('\nvariance C(0,0) by cumulative order:')
for ell in orders:
    print('   %-26s = %.4f' % (order_label(ell), curves[ell][0]))

## 2. Simulation of the SPDE

The conserved nonlinearity enters the spectral integrator as `+g·(−k²)·rfft(φ²)` (the `g_lap`
forcing).  The connected equal-time correlator is averaged over snapshots after burn-in.

In [ ]:
snaps, x_grid, meta = simulate(L=20.0, N=200, mu=mu, D=D, T=T, g_lap=g,
                               dt=0.02, n_steps=110000, burn_in=14000,
                               record_every=10, seed=1)
if not np.all(np.isfinite(snaps)) or np.max(np.abs(snaps)) > 30:
    print('WARNING: the simulation blew up (gradient/conserved stiffness) — '
          'reduce the coupling, raise N, or shrink dt.')
mean = float(np.mean(snaps))                         # ⟨φ⟩ (excess velocity for KPZ; ~0 otherwise)
Cx_full = equal_time_correlator(snaps) - mean**2     # CONNECTED correlator (subtract ⟨φ⟩²)
half = len(x_grid) // 2 + 1
xc, Cx = x_grid[:half], Cx_full[:half]               # one period side, x ≥ 0
print('sim ⟨φ⟩ = %.4f   sim connected C(0,0) = %.4f' % (mean, Cx[0]))
print('theory C(0,0) by cumulative order:')
for ell in orders:
    print('   %-26s = %.4f' % (order_label(ell), curves[ell][0]))

## 3. Compare — theory orders vs simulation

The loop correction to the conserved variance is small (conservation suppression); the structural
signature is the momentum-dependent self-energy, not the scalar $C(0,0)$.

In [ ]:
styles = {0: ('--', 'C0', r'tree  $C_0(x)$'),
          1: ('-',  'C3', 'tree + 1-loop'),
          2: ('-',  'C2', 'tree + 2-loop')}

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for a in ax:
    for ell in orders:
        ls, col, lab = styles[ell]
        a.plot(xs, curves[ell], ls, lw=2, color=col, label=lab)
    a.plot(xc, Cx, 'o', ms=5, color='k', alpha=0.7, label='simulation (connected)')
    a.set_xlabel('x'); a.set_ylabel('C(x, 0)'); a.set_xlim(0, xs.max())
ax[0].set_title('equal-time correlator $C(x,0)$'); ax[0].legend()
ax[1].set_yscale('log'); ax[1].set_title('log scale'); ax[1].legend()
plt.tight_layout(); plt.show()

sim0 = Cx[0]
print('distance |sim − theory| at x=0  (sim connected variance = %.4f):' % sim0)
for ell in orders:
    print('   %-26s : |Δ| = %.4f' % (order_label(ell), abs(sim0 - curves[ell][0])))

## Summary

Model B is the **composite-derivative** representative: the $\nabla^2$ acts on the $\phi^2$ composite,
depositing a momentum form factor $F\propto q^2$ on the loop (`mode='composite'`).  The conservation
law $\Sigma(q\to0)\to0$ emerges from $F=0$ on the tadpole — no model-specific input.

**Knobs:** `MAX_ELL` (loop order), `g` (the conserved coupling), `mu`, `D`, `T`.  The equal-time
variance is conservation-suppressed; the per-$q$ self-energy is the rigorous validation
(`tests/test_full_integrator.py::test_formfactor_bubble_vs_oracle`).